# Day 2: Advanced Data Collection - Web Scraping & APIs 🕷️

**Date:** March 11, 2026  
**Topic:** Web Scraping, API Authentication, Error Handling  
**Duration:** 2-3 hours  

## 🎯 Learning Objectives

By the end of this notebook, you will:
1. ✅ Scrape data from websites using BeautifulSoup
2. ✅ Handle API authentication (API keys)
3. ✅ Implement error handling and retry logic
4. ✅ Manage rate limiting
5. ✅ Combine data from multiple sources
6. ✅ Build a complete data collection pipeline

---

## 📚 Resources for Today

### Documentation
- [BeautifulSoup Docs](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Requests Advanced](https://requests.readthedocs.io/en/latest/user/advanced/)

### Videos (Watch First!)
- [Corey Schafer - Web Scraping](https://www.youtube.com/watch?v=ng2o98k983k) - 30 min ⭐
- [Tech With Tim - Beautiful Soup Tutorial](https://www.youtube.com/watch?v=XVv6mJpFOb0) - 20 min

### Important Notes:
⚠️ **Web Scraping Ethics:**
- Always check `robots.txt` (e.g., https://example.com/robots.txt)
- Respect rate limits
- Don't overload servers
- Check website's Terms of Service
- Use APIs when available (better than scraping)

---

## 🔧 Setup: Install Required Libraries

In [ ]:
# Install required packages
!pip install requests beautifulsoup4 lxml pandas -q

print("✅ All packages installed successfully!")

## 📦 Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
from datetime import datetime
import re
import warnings
warnings.filterwarnings('ignore')

print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n✅ All imports successful!")

---

# Part 1: Web Scraping Basics 🕸️

## Theory: HTML Structure

HTML elements we'll work with:
- `<div>`: Container
- `<table>`: Tables
- `<tr>`: Table row
- `<td>`: Table cell
- `<a>`: Links
- `<h1>`, `<h2>`: Headers
- `class` and `id`: Identifiers

## Exercise 1.1: Basic Web Scraping

Let's scrape a simple webpage.

In [ ]:
# Example HTML (simulating a webpage)
html_content = """
<html>
<head><title>Sample Page</title></head>
<body>
    <h1>Welcome to Data Collection</h1>
    <div class="content">
        <p class="intro">This is a sample webpage.</p>
        <ul>
            <li>Item 1</li>
            <li>Item 2</li>
            <li>Item 3</li>
        </ul>
    </div>
    <table>
        <tr><th>Name</th><th>Age</th><th>City</th></tr>
        <tr><td>Alice</td><td>25</td><td>New York</td></tr>
        <tr><td>Bob</td><td>30</td><td>London</td></tr>
        <tr><td>Charlie</td><td>35</td><td>Paris</td></tr>
    </table>
</body>
</html>
"""

# Parse HTML
soup = BeautifulSoup(html_content, 'html.parser')

# Extract title
title = soup.title.string
print(f"Title: {title}")

# Extract h1
h1 = soup.find('h1').text
print(f"H1: {h1}")

# Extract all list items
items = soup.find_all('li')
print("\nList items:")
for item in items:
    print(f"  - {item.text}")

## Exercise 1.2: Scrape Table to DataFrame

**TASK:** Extract the table and convert to pandas DataFrame.

In [ ]:
# TODO: Your code here
# 1. Find the table
# 2. Extract headers and rows
# 3. Create DataFrame

# Solution:
table = soup.find('table')

# Extract headers
headers = [th.text for th in table.find_all('th')]

# Extract rows
rows = []
for tr in table.find_all('tr')[1:]:  # Skip header row
    cells = [td.text for td in tr.find_all('td')]
    rows.append(cells)

# Create DataFrame
df_table = pd.DataFrame(rows, columns=headers)
print("Extracted Table:")
print(df_table)

## Exercise 1.3: Scrape Real Website - Wikipedia

**TASK:** Scrape a Wikipedia table.

In [ ]:
# Example: Scrape list of countries by population from Wikipedia
url = 'https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population'

# Fetch the page
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

# Find the table (Wikipedia tables have class 'wikitable')
table = soup.find('table', {'class': 'wikitable'})

# Extract data
data = []
if table:
    rows = table.find_all('tr')[1:]  # Skip header
    for row in rows[:10]:  # Get first 10 countries
        cols = row.find_all(['td', 'th'])
        if len(cols) >= 3:
            country = cols[1].text.strip()
            population = cols[2].text.strip()
            data.append([country, population])

# Create DataFrame
df_countries = pd.DataFrame(data, columns=['Country', 'Population'])
print("Top 10 Countries by Population:")
print(df_countries)

**TASK:** Now scrape a different Wikipedia table of your choice!

In [ ]:
# TODO: Your code here
# Ideas: 
# - List of largest cities
# - List of programming languages
# - Olympic medal counts


## Exercise 1.4: Extract Links

**TASK:** Extract all links from a webpage.

In [ ]:
# Example: Extract links from Python.org
url = 'https://www.python.org/'
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

# Find all links
links = soup.find_all('a', href=True)

# Extract href and text
link_data = []
for link in links[:20]:  # First 20 links
    href = link['href']
    text = link.text.strip()
    if text:  # Only if there's text
        link_data.append({'text': text, 'url': href})

df_links = pd.DataFrame(link_data)
print("Links found:")
print(df_links.head(10))

---

# Part 2: API Authentication & Advanced Techniques 🔐

## Exercise 2.1: API with Query Parameters

In [ ]:
# Example: GitHub API with query parameters
url = 'https://api.github.com/search/repositories'

# Parameters
params = {
    'q': 'language:python',
    'sort': 'stars',
    'order': 'desc',
    'per_page': 10
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    repos = data['items']
    
    # Extract relevant info
    repo_info = []
    for repo in repos:
        repo_info.append({
            'name': repo['full_name'],
            'stars': repo['stargazers_count'],
            'forks': repo['forks_count'],
            'language': repo['language'],
            'description': repo['description'][:50] if repo['description'] else 'N/A'
        })
    
    df_repos = pd.DataFrame(repo_info)
    print("Top Python Repositories:")
    print(df_repos)
else:
    print(f"Error: {response.status_code}")

## Exercise 2.2: Pagination - Fetching Multiple Pages

**TASK:** Fetch multiple pages of results from an API.

In [ ]:
# Example: Fetch multiple pages of posts
all_posts = []

for page in range(1, 4):  # Fetch 3 pages
    url = f'https://jsonplaceholder.typicode.com/posts?_page={page}&_limit=10'
    response = requests.get(url)
    
    if response.status_code == 200:
        posts = response.json()
        all_posts.extend(posts)
        print(f"✅ Fetched page {page} ({len(posts)} posts)")
        time.sleep(0.5)  # Be nice to the server
    else:
        print(f"❌ Error on page {page}")

df_all_posts = pd.DataFrame(all_posts)
print(f"\nTotal posts collected: {len(df_all_posts)}")
print(df_all_posts.head())

## Exercise 2.3: Error Handling & Retry Logic

**Important:** Always handle errors when collecting data!

In [ ]:
def fetch_with_retry(url, max_retries=3, delay=1):
    """Fetch URL with retry logic."""
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()  # Raise error for bad status codes
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                print(f"Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print("Max retries reached. Giving up.")
                return None

# Test with a valid URL
url = 'https://jsonplaceholder.typicode.com/users/1'
data = fetch_with_retry(url)

if data:
    print("✅ Data fetched successfully:")
    print(json.dumps(data, indent=2))
else:
    print("❌ Failed to fetch data")

## Exercise 2.4: Rate Limiting

**TASK:** Implement rate limiting to avoid overwhelming servers.

In [ ]:
import time
from datetime import datetime

def rate_limited_fetch(urls, requests_per_second=2):
    """Fetch URLs with rate limiting."""
    delay = 1.0 / requests_per_second
    results = []
    
    for i, url in enumerate(urls):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] Fetching {i+1}/{len(urls)}...")
        
        try:
            response = requests.get(url)
            if response.status_code == 200:
                results.append(response.json())
                print("  ✅ Success")
            else:
                print(f"  ❌ Error: {response.status_code}")
        except Exception as e:
            print(f"  ❌ Exception: {e}")
        
        # Rate limiting
        if i < len(urls) - 1:  # Don't sleep after last request
            time.sleep(delay)
    
    return results

# Test with multiple URLs
urls = [
    'https://jsonplaceholder.typicode.com/users/1',
    'https://jsonplaceholder.typicode.com/users/2',
    'https://jsonplaceholder.typicode.com/users/3',
]

data = rate_limited_fetch(urls, requests_per_second=2)
print(f"\n✅ Fetched {len(data)} results")

---

# Part 3: Combining Multiple Data Sources 🔗

## Exercise 3.1: Merge Data from API and Web Scraping

**TASK:** Combine data from different sources.

In [ ]:
# Source 1: API data (GitHub repos)
url = 'https://api.github.com/search/repositories?q=topic:machine-learning&sort=stars&per_page=5'
response = requests.get(url)
api_data = response.json()['items']

df_api = pd.DataFrame([{
    'name': repo['name'],
    'stars': repo['stargazers_count'],
    'language': repo['language']
} for repo in api_data])

print("Data from API:")
print(df_api)

# Source 2: Manual data (simulating scraped data)
scraped_data = {
    'name': df_api['name'].tolist(),
    'category': ['Deep Learning', 'ML Framework', 'Data Science', 'Neural Networks', 'Computer Vision']
}
df_scraped = pd.DataFrame(scraped_data)

print("\nScraped data:")
print(df_scraped)

# Merge
df_combined = pd.merge(df_api, df_scraped, on='name', how='left')
print("\nCombined data:")
print(df_combined)

---

# 🎯 Project: Build a Complete Data Collection Pipeline

**Your Mission:** Create a pipeline that:
1. Fetches data from an API
2. Scrapes additional data from a website
3. Combines both sources
4. Handles errors gracefully
5. Saves to multiple formats
6. Stores in a database

**Suggestion:** Collect cryptocurrency data
- API: CoinGecko (https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd)
- Scrape: Additional info from CoinMarketCap or similar

In [ ]:
# TODO: Your complete pipeline here
# Step 1: Fetch from API


# Step 2: Scrape website (optional)


# Step 3: Combine data


# Step 4: Clean and process


# Step 5: Save to files


# Step 6: Store in database


## Template: Complete Data Collection Class

In [ ]:
class DataCollector:
    """A reusable data collection class."""
    
    def __init__(self, base_url, rate_limit=2):
        self.base_url = base_url
        self.rate_limit = rate_limit
        self.delay = 1.0 / rate_limit
    
    def fetch_api(self, endpoint, params=None, max_retries=3):
        """Fetch data from API with error handling."""
        url = f"{self.base_url}/{endpoint}"
        
        for attempt in range(max_retries):
            try:
                response = requests.get(url, params=params, timeout=10)
                response.raise_for_status()
                time.sleep(self.delay)  # Rate limiting
                return response.json()
            except Exception as e:
                print(f"Attempt {attempt + 1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)  # Exponential backoff
        return None
    
    def save_data(self, data, filename_prefix):
        """Save data to multiple formats."""
        df = pd.DataFrame(data)
        
        # Save to CSV
        df.to_csv(f'{filename_prefix}.csv', index=False)
        print(f"✅ Saved to {filename_prefix}.csv")
        
        # Save to JSON
        df.to_json(f'{filename_prefix}.json', orient='records', indent=2)
        print(f"✅ Saved to {filename_prefix}.json")
        
        return df

# Example usage:
collector = DataCollector('https://jsonplaceholder.typicode.com')
users = collector.fetch_api('users')

if users:
    df = collector.save_data(users, 'collected_users')
    print("\nData preview:")
    print(df.head())

---

# 📝 Day 2 Summary & Checklist

## What You Learned Today:
- ✅ Web scraping with BeautifulSoup
- ✅ Extracting tables and links from HTML
- ✅ API authentication and query parameters
- ✅ Pagination for multiple pages
- ✅ Error handling and retry logic
- ✅ Rate limiting to be a good citizen
- ✅ Combining data from multiple sources
- ✅ Building reusable data collection pipelines

## Self-Assessment:
Rate yourself (1-5) on:
- [ ] Web scraping: ___/5
- [ ] Error handling: ___/5
- [ ] API advanced features: ___/5
- [ ] Data pipeline design: ___/5

## Practice Projects:
1. **Scrape a job board** (e.g., Indeed, LinkedIn)
2. **Cryptocurrency tracker** (CoinGecko API + scraping)
3. **News aggregator** (NewsAPI + web scraping)
4. **GitHub repository analyzer** (GitHub API + scraping)
5. **Sports statistics** (Sports API + Wikipedia tables)

## Code to Remember:
```python
# Web scraping
soup = BeautifulSoup(html, 'html.parser')
table = soup.find('table')
links = soup.find_all('a')

# Error handling
try:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Error: {e}")

# Rate limiting
time.sleep(1.0 / requests_per_second)
```

---

## 🎯 What's Next?

**Tomorrow (Day 3-4): Data Cleaning**
- Handling missing values
- Dealing with duplicates
- Outlier detection
- Data type conversions
- String cleaning

**Preparation:**
1. Review pandas documentation on missing data
2. Download a messy dataset from Kaggle
3. Watch: [Data Cleaning with Python](https://www.youtube.com/watch?v=... )

---

# 💪 Excellent Work on Day 2!

You've learned advanced data collection techniques. Remember:
1. Always be ethical when scraping
2. Handle errors gracefully
3. Respect rate limits
4. Save your data in multiple formats

**Tasks before tomorrow:**
- [ ] Complete all TODO exercises
- [ ] Build one complete data collection project
- [ ] Update PROGRESS.md
- [ ] Commit code to Git
- [ ] Choose a messy dataset for Day 3

**See you for Day 3 - Data Cleaning!** 🧹✨